<a href="https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1019/semantic_distance_layer_probe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1019 — Layer-wise semantic-distance concept probing

**RQ:** How does a cue's semantic distance affect X-vs-Y internal discriminability under suppression?

Extension of [1015 v3](../1015/white_bear_and_zhou_pilot_v3.ipynb), with eight concept pairs, four graded cue categories and independent semantic-distance estimates. Probe scores are not absolute activation, conscious thoughts, nor evidence of causal reactivation. The target X appears in the instruction. Colab T4 GPU recommended.

Files are saved locally in `GraduationThesis/1019/results/<UTC-run-id>/`. GitHub upload needs the Colab secret `GITHUB_RESULTS_TOKEN`, with repository Contents write permission. A local ZIP is always made, even without the token.


## 0 — Colab setup
Use Runtime → Change runtime type → T4 GPU. Do not reinstall PyTorch; the following removes only optional incompatible audio/vision wheels. Run all cells top-to-bottom.

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers>=4.54,<5" "scikit-learn>=1.3" "pandas>=2.1" "matplotlib>=3.8" "requests>=2.31" "tqdm>=4.65"

In [ ]:
import os, json, re, base64, zipfile, random, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr
from tqdm.auto import tqdm

SEED=42
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
EMBED_ID="sentence-transformers/all-MiniLM-L6-v2"
LAYERS=[4,8,12,16,20,24]   # hidden_states[0]=embedding
LEVELS=["near","medium","far","unrelated"] # hypothetical group labels
CONDITIONS=["neutral","affirm","suppression","replacement"]
MIN_DESCRIPTION_AUC=0.75
RUN_GENERATION=True  # supplemental output analysis; set False for a faster hidden-state-only run
GEN_REPEATS=1
UPLOAD_TO_GITHUB=True
REPO="ryouy/GraduationThesis"
BRANCH="main"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
if DEVICE=="cpu":print("WARNING: No GPU; switch Colab runtime to T4 for practical speed.")
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BASE=Path("/content/GraduationThesis") if Path("/content").is_dir() else Path.cwd()/"GraduationThesis"
OUT=BASE/"1019"/"results"/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
def save_csv(frame,name):
    frame.to_csv(OUT/name,index=False)
    print("Saved",name,len(frame),"rows")
def save_plot(name):
    plt.tight_layout();plt.savefig(OUT/name,dpi=180,bbox_inches="tight");plt.show();plt.close()
print("Device:",DEVICE,"Results:",OUT)

## 1 — Cue bank (fixed before modeling)

Eight concept pairs inherited from 1015 v3. For each X, four proposed distance bins contain **two distinct English sentences**. All cue sentences are checked not to contain the literal X or Y. The bins mix descriptive and associative relations and must not be treated as validated semantic distances.

In [ ]:
# Compact data: x, y, three x-descriptions, three y-descriptions, and 4 × 2 cue sentences.
ITEMS=[
("banana","apple",
 ["a long sweet yellow produce item","a soft fruit that is peeled before eating","a curved tropical snack fruit"],
 ["a crisp round orchard fruit","a red or green fruit with a core","a firm tree-grown fruit used in pies"],
 [
 ["The shopper peeled a long yellow fruit for a snack.","A curved yellow fruit lay beside the cereal bowl."],
 ["A monkey searched the canopy for something sweet to eat.","A smoothie recipe called for ripe tropical fruit."],
 ["The rainforest canopy was full of small birds.","A stall displayed imported produce in the morning."],
 ["A clerk stamped documents at a railway office.","Workers repaired a cracked pavement outside the library."]]),
("elephant","giraffe",
 ["a giant gray mammal with a long flexible nose","a tusked herbivore with exceptionally large ears","a heavy savanna animal famous for its trunk"],
 ["a tall spotted animal with an unusually long neck","a long-legged hoofed mammal browsing high branches","a graceful African animal with patterned patches"],
 [
 ["A giant gray mammal swung its trunk by a waterhole.","Huge ears flapped as a tusked animal crossed the plain."],
 ["A safari guide discussed the world's largest land mammals.","A herd of enormous herbivores crossed the savanna."],
 ["Tourists visited a wildlife reserve before lunch.","Grasslands stretched toward distant rocky hills."],
 ["A librarian replaced several damaged catalog cards.","Subway timetables changed after maintenance."]]),
("tiger","lion",
 ["an orange Asian wild cat marked by black stripes","a solitary jungle predator with dark body bands","a large striped feline in the forest"],
 ["a tawny big cat with a prominent mane","a social wild feline that lives in a pride","a large African carnivore famous for its mane"],
 [
 ["Orange fur with black stripes moved between the trees.","A solitary striped big cat stalked prey in dense forest."],
 ["A powerful wild feline prowled through an Asian jungle.","Zookeepers fed a large carnivorous cat."],
 ["The rainforest path was damp after a storm.","A wildlife photographer adjusted a forest camera."],
 ["A passenger checked the terminal departure board.","A contractor installed pipes beneath a road."]]),
("piano","violin",
 ["a keyboard instrument with black and white keys","an acoustic instrument whose strings are struck by hammers","a large instrument played using both hands on keys"],
 ["a bowed orchestral instrument held near the shoulder","a small four-string instrument played with a bow","a high pitched classical string instrument"],
 [
 ["A musician pressed black and white keys across the keyboard.","A grand acoustic keyboard instrument stood on stage."],
 ["A student practiced chords in the music room.","The hall hosted a classical solo performance."],
 ["The audience entered the theater before the lights dimmed.","A composer examined pages of printed notation."],
 ["A mechanic repaired the engine of a bus.","Rainwater collected behind the garage."]]),
("coffee","tea",
 ["a dark beverage made from ground roasted beans","an espresso-based caffeinated drink","a bitter aromatic brew served in cafes"],
 ["an infusion made by steeping dried leaves","a leaf-based hot drink","a drink brewed from plant leaves rather than roasted beans"],
 [
 ["Roasted beans were ground before hot water was poured.","A bitter dark drink steamed from an espresso cup."],
 ["A barista prepared an order in the morning cafe.","Freshly brewed drinks filled a small shop with aroma."],
 ["Two friends sat at a table downtown.","A commuter stopped for breakfast on the way to work."],
 ["An engineer inspected a bridge's steel supports.","Children sorted pebbles by a dry creek."]]),
("ocean","river",
 ["a vast saltwater body spanning continents","an immense stretch of open seawater","a saline expanse where large ships sail"],
 ["a flowing freshwater channel through land","a watercourse moving between banks","a long inland current that runs toward the sea"],
 [
 ["Saltwater waves rolled toward an endless horizon.","A vast expanse of seawater separated continents."],
 ["A ship sailed beyond the coastline toward open water.","A sailor studied tides before departing from port."],
 ["A coastal village prepared for tourist season.","An old lighthouse stood on a rocky cliff."],
 ["A teacher rearranged desks before class.","A bakery sold fresh bread at dawn."]]),
("castle","palace",
 ["a defensive medieval stone fortress","a stronghold with battlements and thick walls","a fortified old building with a moat and towers"],
 ["an ornate royal residence with grand halls","an opulent residence used by a monarch","a luxurious court building for official ceremonies"],
 [
 ["A medieval fortress had thick walls and tall watchtowers.","Battlements and a deep moat surrounded the stronghold."],
 ["Knights returned to a fortified royal residence.","Visitors toured an old stone keep guarded by soldiers."],
 ["A medieval village held a street festival.","An ancient hillside settlement welcomed tourists."],
 ["A dentist checked the appointment register.","A gardener planted seedlings in spring."]]),
("bicycle","scooter",
 ["a pedal-powered two-wheeled road vehicle","a human powered machine with a saddle and chain","a wheeled vehicle propelled by turning foot pedals"],
 ["a small footboard vehicle steered by handlebars","a compact kick-powered personal vehicle","a two wheeled standing device with a narrow platform"],
 [
 ["A rider pedaled a two-wheeled vehicle along the lane.","A metal frame with pedals and handlebars leaned against a wall."],
 ["A commuter wore a helmet on the morning ride.","The repair shop adjusted a chain and brake cables."],
 ["A city built lanes beside a busy street.","A fitness club hosted an outdoor endurance event."],
 ["A chef simmered vegetables in a soup pot.","A violinist rehearsed a short piece backstage."]])
]
assert len(ITEMS)==8
def has_word(s,w):
    return bool(re.search(r"(?<![A-Za-z])"+re.escape(w)+r"(?![A-Za-z])",s,re.I))
cue_rows=[]
for x,y,xd,yd,level_sets in ITEMS:
    assert len(xd)==len(yd)==3 and len(level_sets)==4
    for level,sentences in zip(LEVELS,level_sets):
        assert len(sentences)==2
        for cue_id,s in enumerate(sentences,1):
            assert not has_word(s,x) and not has_word(s,y),(x,s)
            cue_rows.append(dict(concept=x,replacement=y,level=level,
                                 cue_id=cue_id,cue_text=s))
cues=pd.DataFrame(cue_rows)
assert len(cues)==64
display(cues.head(6))

## 2 — Independent semantic distance (MiniLM)

Each cue's distance to X is estimated as one minus the cosine similarity between a separate MiniLM encoder's embeddings of a target phrase and the cue sentence. **Proposed near/medium/far/unrelated groups are not factual ground truth.** Order failures are logged. The continuous measured distance is the primary predictor; human ratings should be collected before making stronger claims.

In [ ]:
embed_tok=AutoTokenizer.from_pretrained(EMBED_ID,trust_remote_code=False)
embed_model=AutoModel.from_pretrained(EMBED_ID,trust_remote_code=False).cpu().eval()
concepts=[item[0] for item in ITEMS]
texts=[f"This text concerns {x}." for x in concepts]+cues.cue_text.tolist()
vecs=[]
for start in tqdm(range(0,len(texts),32),desc="MiniLM embeddings"):
    batch=embed_tok(texts[start:start+32],return_tensors="pt",padding=True,truncation=True,max_length=128)
    with torch.inference_mode():
        h=embed_model(**batch).last_hidden_state
        mask=batch["attention_mask"].unsqueeze(-1)
        p=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
        vecs.append(F.normalize(p.float(),dim=-1).cpu().numpy())
vecs=np.concatenate(vecs,axis=0)
reference={concept:vecs[i] for i,concept in enumerate(concepts)}
cues["cosine_similarity"]=[float(reference[row.concept]@vecs[len(concepts)+idx])
                           for idx,row in cues.reset_index(drop=True).iterrows()]
cues["semantic_distance"]=1-cues.cosine_similarity
cues["word_count"]=cues.cue_text.str.split().str.len()
save_csv(cues,"01_cue_distances.csv")
level_distance=cues.groupby(["concept","level"],as_index=False).semantic_distance.mean()
order=level_distance.pivot(index="concept",columns="level",values="semantic_distance").reindex(columns=LEVELS)
order["ordered_as_proposed"]=np.all(np.diff(order[LEVELS].values,axis=1)>0,axis=1)
save_csv(order.reset_index(),"02_distance_order_diagnostics.csv")
display(order.round(3))
print("Distance levels ordered for",int(order.ordered_as_proposed.sum()),"/",len(order),"concepts.")
if not order.ordered_as_proposed.all():
    print("WARNING: Proposed groups are not consistently distance ordered. Continuous measured distances take precedence.")
del embed_model,embed_tok,vecs
if torch.cuda.is_available():torch.cuda.empty_cache()

## 3 — Load Qwen and match the hidden-state readout position

The hidden vector is taken at the **last token of the assistant-generation header** added by Qwen's chat template. That is the same structural token across probe training and all test prompts. Although we append a common `Topic:` marker in the user text, the actual measured token comes *after* that marker. Variable input length and differences between instruction wording remain potential confounders.

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_ID,dtype=DTYPE,trust_remote_code=False,low_cpu_mem_usage=True).to(DEVICE).eval()
assert model.config.num_hidden_layers==24
assert max(LAYERS)<=model.config.num_hidden_layers
def chat(s):
    return tok.apply_chat_template([{"role":"user","content":s}],
                                   tokenize=False,add_generation_prompt=True)
@torch.inference_mode()
def features(prompt):
    enc=tok(chat(prompt),return_tensors="pt",add_special_tokens=False).to(DEVICE)
    hs=model(**enc,output_hidden_states=True,return_dict=True,use_cache=False).hidden_states
    return {layer:hs[layer][0,-1,:].float().cpu().numpy() for layer in LAYERS},int(enc.input_ids.shape[1])
testfeat,testlen=features("The topic word is banana.\nTopic:")
assert len(testfeat)==len(LAYERS) and all(np.isfinite(v).all() for v in testfeat.values())
print("Model ready:",MODEL_ID,"input tokens:",testlen,"hidden dimension:",testfeat[24].size)

## 4 — Train independent X-vs-Y probes and validate on unseen descriptions

Train 8 named templates × 2 classes = 16 examples per concept. Validate separately on 3 novel named templates × 2 classes, and on 3 **different name-free descriptions per class** × 2 held-out frames = 12 evaluation contexts. Descriptions are related paraphrases, not 12 statistically independent semantic concepts. No test cues are used in probe fitting. AUC ≥ 0.75 is the prespecified exploratory screen, and failing probes' data are **still saved**.

In [ ]:
TRAIN_FRAMES=[
"The note concerns {}.\nTopic:","Think of {}.\nTopic:",
"An example topic is {}.\nTopic:","Someone mentioned {} earlier.\nTopic:",
"The main subject is {}.\nTopic:","The concept to discuss is {}.\nTopic:",
"Keep {} in mind.\nTopic:","Focus attention on {}.\nTopic:"]
NAMED_TEST_FRAMES=[
"A writer spoke about {}.\nTopic:","The conversation revolves around {}.\nTopic:",
"The word of interest is {}.\nTopic:"]
DESC_TEST_FRAMES=["An exhibit concerns {}.\nTopic:","A document describes {}.\nTopic:"]
probe_inputs=[]
for x,y,x_desc,y_desc,_ in tqdm(ITEMS,desc="Probe fit and holdout inputs"):
    for label,word,descriptions in [(1,x,x_desc),(0,y,y_desc)]:
        for f in TRAIN_FRAMES:
            p=f.format(word);h,n=features(p)
            probe_inputs.append(dict(concept=x,split="train",label=label,prompt=p,features=h))
        for f in NAMED_TEST_FRAMES:
            p=f.format(word);h,n=features(p)
            probe_inputs.append(dict(concept=x,split="named_holdout",label=label,prompt=p,features=h))
        for desc in descriptions:
            assert not has_word(desc,x) and not has_word(desc,y),(x,desc)
            for f in DESC_TEST_FRAMES:
                p=f.format(desc);h,n=features(p)
                probe_inputs.append(dict(concept=x,split="description_holdout",label=label,prompt=p,features=h))
probe_inputs=pd.DataFrame(probe_inputs)
assert probe_inputs.groupby("split").size().to_dict()=={
    "train":128,"named_holdout":48,"description_holdout":96}
probes={};metrics=[]
for x in concepts:
    ex=probe_inputs[probe_inputs.concept==x]
    train=ex[ex.split=="train"]
    probes[x]={}
    for layer in LAYERS:
        X=np.stack([a[layer] for a in train.features]);y=train.label.to_numpy()
        clf=make_pipeline(StandardScaler(),LogisticRegression(
            C=0.1,solver="liblinear",random_state=SEED,max_iter=2000))
        clf.fit(X,y)
        sd=max(float(clf.decision_function(X).std(ddof=1)),1e-8)
        probes[x][layer]=(clf,sd)
        for split in ["named_holdout","description_holdout"]:
            ev=ex[ex.split==split];V=np.stack([a[layer] for a in ev.features])
            score=clf.decision_function(V)
            metrics.append(dict(concept=x,layer=layer,split=split,n=len(ev),
                auc=float(roc_auc_score(ev.label.to_numpy(),score)),
                accuracy=float(((score>=0).astype(int)==ev.label.to_numpy()).mean()),
                train_decision_sd=sd))
validation=pd.DataFrame(metrics)
valid=validation[validation.split=="description_holdout"].copy()
valid["validated"]=valid.auc>=MIN_DESCRIPTION_AUC
save_csv(validation,"03_probe_validation.csv")
save_csv(valid,"04_probe_validated_flags.csv")
display(valid.groupby("layer").agg(mean_auc=("auc","mean"),passed=("validated","sum"),total=("validated","size")).round(3))
del probe_inputs
print("Do not interpret scores from layers with poor held-out validation as semantic concept activation.")

## 5 — Main experiment: four instruction conditions × graded cues × selected layers

Use the exact 1015-v3 instruction sentences. Suppression literally repeats X and is longer than Neutral, so interpretation requires caution: not every difference is attributable exclusively to suppression. The no-cue reference is also measured for each instruction. **The prompt and the terminal token are held constant within a paired condition/cue comparison except for the manipulated instruction or cue.**

In [ ]:
def instruction(x,y,c):
    return {
        "neutral":f"The topic word is {x}.",
        "affirm":f"Think of {x}.",
        "suppression":f"Do not think of {x}. Do not write the word {x}.",
        "replacement":f"Think of {y} instead of {x}. Do not write the word {x}."
    }[c]
def probe_prompt(x,y,c,cue=None):
    return instruction(x,y,c)+"\n"+(cue+"\n" if cue else "")+"Topic:"

validation_map={(row.concept,int(row.layer)):(float(row.auc),bool(row.validated))
                for row in valid.itertuples()}
measure=[]
for x,y,_,_,_ in tqdm(ITEMS,desc="Main experiment"):
    cue_cases=[dict(level="none",cue_id=0,cue_text="",semantic_distance=np.nan)]
    cue_cases+=cues[cues.concept==x][["level","cue_id","cue_text","semantic_distance"]].to_dict("records")
    for cond in CONDITIONS:
        for row in cue_cases:
            p=probe_prompt(x,y,cond,row["cue_text"])
            feat,n=features(p)
            for layer in LAYERS:
                clf,train_sd=probes[x][layer]
                raw=float(clf.decision_function(feat[layer].reshape(1,-1))[0])
                auc,is_valid=validation_map[(x,layer)]
                measure.append(dict(concept=x,replacement=y,condition=cond,
                    level=row["level"],cue_id=row["cue_id"],cue_text=row["cue_text"],
                    semantic_distance=row["semantic_distance"],layer=layer,
                    prompt_tokens=n,prompt=p,probe_raw_logodds=raw,
                    probe_standardized=raw/train_sd,train_decision_sd=train_sd,
                    validation_description_auc=auc,validated=is_valid))
scores=pd.DataFrame(measure)
assert len(scores)==len(ITEMS)*len(CONDITIONS)*(1+len(LEVELS)*2)*len(LAYERS)
base_scores=scores[scores.level=="none"][["concept","condition","layer","probe_standardized"]].rename(
    columns={"probe_standardized":"no_cue_standardized"})
scores=scores.merge(base_scores,on=["concept","condition","layer"],validate="many_to_one")
scores["delta_from_no_cue"]=scores.probe_standardized-scores.no_cue_standardized
save_csv(scores,"05_probe_scores_all_conditions.csv")
lengths=scores.groupby(["condition","level"],as_index=False).prompt_tokens.agg(["mean","min","max"])
save_csv(lengths,"06_prompt_length_diagnostics.csv")
display(lengths)

## 6 — Matched condition contrasts and DID

For a concept X and cue c, let S_c be the suppression X/Y probe score and N_c the neutral score, with u denoting the **average of two unrelated cues for the same concept**. The target contrast is **D=(S_c−S_u)−(N_c−N_u)**. A positive D does not mean the suppression score is higher than neutral: it only means a larger cue-versus-unrelated difference. Standardize each independently trained probe score using its training decision-score SD before averaging across concepts.

In [ ]:
cued=scores[scores.level!="none"].copy()
index_cols=["concept","replacement","level","cue_id","cue_text","semantic_distance","layer",
            "validated","validation_description_auc"]
wide=cued.pivot(index=index_cols,columns="condition",
                values=["probe_raw_logodds","probe_standardized","delta_from_no_cue"]).reset_index()
wide.columns=["_".join(str(p) for p in entry if p!="") for entry in wide.columns]
wide["suppression_minus_neutral_raw"]=wide.probe_raw_logodds_suppression-wide.probe_raw_logodds_neutral
wide["suppression_minus_neutral_std"]=wide.probe_standardized_suppression-wide.probe_standardized_neutral
wide["replacement_minus_suppression_std"]=wide.probe_standardized_replacement-wide.probe_standardized_suppression
anchors=wide[wide.level=="unrelated"].groupby(["concept","layer"],as_index=False).agg(
    unrelated_diff_raw=("suppression_minus_neutral_raw","mean"),
    unrelated_diff_std=("suppression_minus_neutral_std","mean"))
wide=wide.merge(anchors,on=["concept","layer"],validate="many_to_one")
wide["DID_raw"]=wide.suppression_minus_neutral_raw-wide.unrelated_diff_raw
wide["DID_std"]=wide.suppression_minus_neutral_std-wide.unrelated_diff_std
assert wide[wide.level=="unrelated"].groupby(["concept","layer"]).DID_std.mean().abs().max()<1e-8
save_csv(wide,"07_cue_level_paired_contrasts.csv")
by_concept=wide.groupby(["concept","layer","level","validated"],as_index=False).agg(
    mean_distance=("semantic_distance","mean"),mean_DID_std=("DID_std","mean"),
    mean_DID_raw=("DID_raw","mean"),mean_suppression=("probe_standardized_suppression","mean"),
    mean_neutral=("probe_standardized_neutral","mean"),n_cues=("cue_id","size"))
save_csv(by_concept,"08_per_concept_layer_summary.csv")
display(by_concept[by_concept.layer==24].pivot(index="concept",columns="level",values="mean_DID_std").reindex(columns=LEVELS).round(3))

## 7 — Independent unit = concept; concept-cluster bootstrap intervals

Every concept contributes two distinct cues per distance group. These cues are **not** treated as independent concepts. Bootstrap over 8 concept IDs; do not claim strong population inference from these small exploratory confidence intervals. Means of failing probes are retained in an all-data diagnostic, but primary plots only aggregate probes whose held-out description ROC-AUC is ≥ 0.75.

In [ ]:
def concept_boot(values,seed=SEED,n_boot=3000):
    a=np.asarray(values,float)
    if len(a)<3:
        return float(a.mean()) if len(a) else np.nan,np.nan,np.nan
    rng=np.random.default_rng(seed)
    samples=rng.choice(a,(n_boot,len(a)),replace=True).mean(axis=1)
    return float(a.mean()),float(np.quantile(samples,.025)),float(np.quantile(samples,.975))
aggregates=[]
for layer in LAYERS:
    for level in LEVELS:
        for group in ["all","validated"]:
            z=by_concept[(by_concept.layer==layer)&(by_concept.level==level)]
            if group=="validated":z=z[z.validated]
            m,lo,hi=concept_boot(z.mean_DID_std.to_numpy(),seed=SEED+layer*7+LEVELS.index(level))
            aggregates.append(dict(layer=layer,level=level,analysis=group,
                n_concepts=int(len(z)),mean_DID_std=m,ci95_lower=lo,ci95_upper=hi))
aggregates=pd.DataFrame(aggregates)
save_csv(aggregates,"09_DID_concept_bootstrap.csv")
display(aggregates[(aggregates.analysis=="validated")&(aggregates.level=="near")].round(3))

## 8 — Result figures

AUC heatmap first (assess interpretability), then paired instruction effects by layer, suppression-specific DID across layers, and independent continuous semantic distance vs DID at the final layer. Each figure is saved in the same 1019 run folder as the CSV. The plotted estimates are **new results only after executing this notebook**, not the 1015 pilot figures.

In [ ]:
plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False})
colors={"near":"#184e77","medium":"#2786aa","far":"#7fa9bc","unrelated":"#929292"}
# Fig 1: concept- and layer-specific independent description AUC
mat=valid.pivot(index="concept",columns="layer",values="auc").reindex(index=concepts,columns=LAYERS)
fig,ax=plt.subplots(figsize=(8.0,4.8))
pic=ax.imshow(mat.values,cmap="Blues",vmin=0,vmax=1,aspect="auto")
ax.set(xticks=np.arange(len(LAYERS)),xticklabels=LAYERS,
       yticks=np.arange(len(concepts)),yticklabels=concepts,
       xlabel="Hidden-state layer",title="Probe ROC-AUC: name-free description holdout")
for i in range(len(concepts)):
    for j in range(len(LAYERS)):
        v=mat.iloc[i,j]
        ax.text(j,i,f"{v:.2f}",fontsize=8,ha="center",va="center",
                color="white" if v>=0.80 else "black")
fig.colorbar(pic,ax=ax,label="ROC-AUC")
save_plot("01_probe_validation_auc.png")

# Fig 2: within-condition cue delta vs own no-cue baseline; concept is mean unit.
fig,axs=plt.subplots(1,2,figsize=(12,4.6),sharey=True)
for ax,cond in zip(axs,["neutral","suppression"]):
    for level in LEVELS:
        q=cued[(cued.condition==cond)&(cued.level==level)&(cued.validated)]
        q=q.groupby(["concept","layer"],as_index=False).delta_from_no_cue.mean()
        if not len(q):continue
        q=q.groupby("layer").delta_from_no_cue.mean()
        ax.plot(q.index,q.values,"-o",color=colors[level],label=level)
    ax.axhline(0,c="0.65",ls="--",lw=1)
    ax.set(xlabel="Hidden-state layer",title=cond.capitalize())
    ax.grid(alpha=.20)
axs[0].set_ylabel("X/Y probe delta from no cue (training SD units)")
axs[1].legend(title="Proposed cue group")
fig.suptitle("Layer-wise cue effects: validated concept probes only")
save_plot("02_neutral_vs_suppression_layerwise.png")

# Fig 3: DID vs unrelated by layer. Filter out layers with fewer than 3 valid concept probes.
fig,ax=plt.subplots(figsize=(8.6,4.8))
for level in LEVELS:
    q=aggregates[(aggregates.analysis=="validated")&(aggregates.level==level)&(aggregates.n_concepts>=3)]
    if not len(q):continue
    ax.plot(q.layer,q.mean_DID_std,"-o",color=colors[level],label=level)
    ax.fill_between(q.layer.to_numpy(dtype=float),q.ci95_lower.to_numpy(dtype=float),
                    q.ci95_upper.to_numpy(dtype=float),alpha=.12,color=colors[level])
ax.axhline(0,c="0.55",ls="--")
ax.set(xlabel="Hidden-state layer",ylabel="Suppression-specific DID (standardized)",
       title="DID vs unrelated cues; concept bootstrap 95% intervals")
ax.grid(alpha=.20)
if ax.get_legend_handles_labels()[1]:ax.legend(title="Proposed cue group")
save_plot("03_layerwise_DID.png")

# Fig 4: continuous independently measured distance, final layer.
last=wide[(wide.layer==max(LAYERS)) & (wide.validated)].copy()
fig,ax=plt.subplots(figsize=(8.6,4.7))
for level in LEVELS:
    q=last[last.level==level]
    if len(q):ax.scatter(q.semantic_distance,q.DID_std,color=colors[level],label=level,alpha=.8,s=38)
ax.axhline(0,c="0.55",ls="--")
ax.set(xlabel="Semantic distance = 1 - MiniLM cosine",
       ylabel="DID (training SD units)",title="Independent semantic distance vs final-layer internal contrast")
ax.grid(alpha=.2)
if ax.get_legend_handles_labels()[1]:ax.legend(title="Proposed cue group")
save_plot("04_continuous_distance_vs_DID.png")

## 9 — Exploratory within-concept distance trends

Compute distance–DID slope and Spearman correlation separately for each concept at the final layer, then summarize the eight concept-level slopes. Only validated concept probes are included. Cue-level p-values are **descriptive**, because paraphrases are dependent and concept samples are few.

In [ ]:
slopes=[]
for x,q in last.groupby("concept"):
    if len(q)<4:continue
    xs=q.semantic_distance.to_numpy(float);ys=q.DID_std.to_numpy(float)
    rho,p=spearmanr(xs,ys)
    xc=xs-xs.mean();yc=ys-ys.mean()
    slope=float((xc@yc)/(xc@xc)) if xc@xc>1e-10 else np.nan
    slopes.append(dict(concept=x,n_cues=len(q),spearman_rho=float(rho),
                       exploratory_spearman_p=float(p),within_concept_slope=slope))
slopes=pd.DataFrame(slopes,columns=["concept","n_cues","spearman_rho",
                                    "exploratory_spearman_p","within_concept_slope"])
save_csv(slopes,"10_within_concept_distance_trends.csv")
if len(slopes):
    m,lo,hi=concept_boot(slopes.within_concept_slope.dropna().to_numpy())
    print(f"Exploratory average slope: {m:.3f} (concept bootstrap 95% [{lo:.3f}, {hi:.3f}]); n={len(slopes)} concepts.")
    display(slopes.round(3))
else:
    print("No validated final-layer probes with adequate cue samples: do not interpret distance trends.")

## 10 — Optional behavior: target-word mentions in separately prompted generations

The behavior prompt ends in a different instruction than the probe readout. Thus generation frequencies do **not** align 1:1 with a specific hidden state measured above. Enable RUN_GENERATION at the top before Run all if needed. With one sample per cue, the estimated mention rates have substantial sampling uncertainty.

In [ ]:
@torch.inference_mode()
def sample_answer(prompt,seed):
    torch.manual_seed(int(seed))
    tokens=tok(chat(prompt),return_tensors="pt",add_special_tokens=False).to(DEVICE)
    output=model.generate(**tokens,max_new_tokens=40,do_sample=True,
                          temperature=.8,top_p=.95,pad_token_id=tok.eos_token_id)
    return tok.decode(output[0,tokens.input_ids.shape[1]:],skip_special_tokens=True).strip()
PLURALS={"banana":"bananas","elephant":"elephants","tiger":"tigers","piano":"pianos",
         "coffee":"coffees","ocean":"oceans","castle":"castles","bicycle":"bicycles"}
if RUN_GENERATION:
    generated=[]
    for x,y,_,_,_ in tqdm(ITEMS,desc="Optional generation"):
        sub=cues[cues.concept==x]
        for cond in ["neutral","suppression"]:
            for row in sub.itertuples():
                p=instruction(x,y,cond)+"\n"+row.cue_text+"\nWrite one short sentence about whatever comes to mind."
                gh,_=features(p)  # same prompt used for the following generation
                gc,gsd=probes[x][max(LAYERS)]
                gen_score=float(gc.decision_function(gh[max(LAYERS)].reshape(1,-1))[0])/gsd
                is_valid=bool(valid[(valid.concept==x)&(valid.layer==max(LAYERS))].validated.iloc[0])
                for rep in range(GEN_REPEATS):
                    seed=SEED+concepts.index(x)*10000+LEVELS.index(row.level)*1000+row.cue_id*100+rep
                    answer=sample_answer(p,seed)
                    mention=int(has_word(answer,x) or has_word(answer,PLURALS[x]))
                    generated.append(dict(concept=x,condition=cond,level=row.level,
                        cue_id=row.cue_id,replicate=rep,semantic_distance=row.semantic_distance,
                        target_mentioned=mention,generation_prompt_probe_std=gen_score,
                        generation_prompt_probe_validated=is_valid,response=answer,prompt=p))
    generated=pd.DataFrame(generated)
    save_csv(generated,"11_optional_generations.csv")
    rates=generated.groupby(["concept","condition","level"],as_index=False).target_mentioned.agg(["mean","sum","count"])
    save_csv(rates,"12_optional_mention_rates_by_concept.csv")
    rr=generated.groupby(["condition","level"],as_index=False).target_mentioned.mean()
    save_csv(rr,"13_optional_pooled_mention_rates.csv")
    # Exploratory cross-modal cases, measured at the EXACT prompt that generated each answer.
    # This prompt differs from the Topic:-anchored probe training, so distribution shift remains.
    sg=generated[(generated.condition=="suppression")&(generated.generation_prompt_probe_validated)].copy()
    if len(sg):
        sg["within_concept_top_quartile"]=sg.generation_prompt_probe_std>=sg.groupby("concept").generation_prompt_probe_std.transform(lambda z:z.quantile(.75))
        retained=sg[(sg.within_concept_top_quartile)&(sg.target_mentioned==0)]
        save_csv(retained,"14_high_ranked_probe_no_target_word.csv")
        print("High-ranked suppression prompt probe with no literal target word:",len(retained),"cases; this is NOT proof of absolute activation.")
        fig,ax=plt.subplots(figsize=(8,4.4))
        for mentioned,label in [(0,"No target word"),(1,"Target word present")]:
            part=sg[sg.target_mentioned==mentioned]
            ax.scatter(part.semantic_distance,part.generation_prompt_probe_std,s=40,alpha=.7,label=label)
        ax.set(xlabel="Independent semantic distance (1 - MiniLM cosine)",ylabel="Generation-prompt X/Y probe score (training SD units)",
            title="Suppression: internal readout vs generated literal word (exploratory)")
        ax.legend();ax.grid(alpha=.15);save_plot("06_generation_prompt_score_vs_mentions.png")
    fig,ax=plt.subplots(figsize=(7.5,4))
    xloc=np.arange(len(LEVELS));width=.36
    for shift,cond in [(-width/2,"neutral"),(width/2,"suppression")]:
        q=rr[rr.condition==cond].set_index("level").reindex(LEVELS)
        ax.bar(xloc+shift,100*q.target_mentioned,width,label=cond)
    ax.set(xticks=xloc,xticklabels=LEVELS,ylim=(0,100),
        ylabel="Exact target mention (%)",title="Supplemental stochastic generations")
    ax.legend()
    save_plot("05_optional_behavior_mentions.png")
else:
    print("Generation skipped. Main concept-probe experiment and four figures are complete.")

## 11 — Sanity checks and run manifest

Results are exploratory. Prompt length and repeated literal X mentions differ between neutral and suppression instructions. Validation of the probe on name-free descriptions reduces but does not eliminate leakage/confounding. No intervening model-state modification or causal intervention has been conducted.

In [ ]:
assert len(cues)==8*4*2 and cues.semantic_distance.notna().all()
assert len(scores)==8*4*9*len(LAYERS)
assert len(wide)==8*8*len(LAYERS)
assert len(valid)==8*len(LAYERS)
assert not scores.probe_standardized.isna().any()
assert not wide.DID_std.isna().any()
run_metadata={
 "title":"1019 semantic distance / suppressed concept layer probe",
 "run_id":RUN_ID,"model":MODEL_ID,"semantic_distance_model":EMBED_ID,
 "seed":SEED,"conditions":CONDITIONS,"layers":LAYERS,
 "n_concepts":len(ITEMS),"n_cues_each_level":2,"levels_are_proposed":LEVELS,
 "distance_formula":"1 - cosine(MiniLM sentence embeddings)",
 "semantic_order_pass_count":int(order.ordered_as_proposed.sum()),
 "validation_description_examples_per_pair":12,
 "auc_cutoff":MIN_DESCRIPTION_AUC,
 "probe_accepted_by_layer":{str(l):int(z.validated.sum()) for l,z in valid.groupby("layer")},
 "run_generation":RUN_GENERATION,"generation_repeats":GEN_REPEATS,
 "device":DEVICE,"torch":torch.__version__,
 "transformers":__import__("transformers").__version__,
 "python":sys.version,"completed_utc":datetime.now(timezone.utc).isoformat(),
 "limitations":[
   "Probe is relative X/Y discriminability, not absolute concept activation.",
   "Target X appears in every instruction; lexical persistence is not tested.",
   "Related/unrelated comparisons are between separate prompts, not temporal neural reactivation.",
   "Suppression repeats X twice while the neutral instruction names X once.",
   "Preselected levels are subjective and may disagree with independent MiniLM distances.",
   "Only 8 target concepts; bootstrap treats concepts, not cue texts, as sampling units.",
   "Description evaluation uses only three distinct semantic descriptions per class.",
   "No causal steering, no tested external model generalization.",
   "Behavior mode uses different final instructions than probe mode."
 ]
}
(OUT/"metadata.json").write_text(json.dumps(run_metadata,indent=2,ensure_ascii=False),encoding="utf-8")
result_readme="\n".join([
 "# 1019 Semantic-distance × suppression run","",
 "Run ID: "+RUN_ID,"Model: "+MODEL_ID,"",
 "## Interpretation",
 "X-vs-Y probe decision scores are not activation probabilities.",
 "Positive DID means a larger related-cue vs unrelated-cue contrast under suppression than neutral.",
 "The neutral and suppression prompts differ in the number of literal X mentions.",
 "Semantic bins are hypotheses; use 01_cue_distances.csv and 02_distance_order_diagnostics.csv.",
 "Only validated probes (description AUC >= 0.75) are used in the main figures.",
 "Unfiltered raw scores are preserved. Concept count n=8; bootstrap intervals exploratory.","",
 "## Files",
 "01_cue_distances.csv - cue text, lexical checks, independent distance",
 "02_distance_order_diagnostics.csv - ordering of candidate distance groups",
 "03_probe_validation.csv - all held-out probes",
 "04_probe_validated_flags.csv - semantically validated probes",
 "05_probe_scores_all_conditions.csv - all 4 conditions and baseline by layer",
 "06_prompt_length_diagnostics.csv - remaining length confounds",
 "07_cue_level_paired_contrasts.csv - within-cue suppression vs neutral",
 "08_per_concept_layer_summary.csv - cluster-level aggregation",
 "09_DID_concept_bootstrap.csv - exploratory 95% concept intervals",
 "10_within_concept_distance_trends.csv - final-layer continuous-distance analysis",
 "01-04_*.png - visualization; optional 05 behavior graphic",
 "metadata.json - run configuration and all limitations",
 "","Original code: 1019/semantic_distance_layer_probe.ipynb"
])+"\n"
(OUT/"README.md").write_text(result_readme,encoding="utf-8")
zip_path=OUT.parent/(RUN_ID+"_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(OUT.iterdir()):
        if p.is_file():archive.write(p,arcname=f"1019/results/{RUN_ID}/{p.name}")
print("Saved CSV:",len(list(OUT.glob("*.csv"))),"Saved PNG:",len(list(OUT.glob("*.png"))))
print("Local archive:",zip_path,"bytes:",zip_path.stat().st_size)

## 12 — Upload every CSV, PNG, JSON, and README into GitHub's 1019 folder

To save Colab results back to the original GitHub repository, open Colab Secrets, set `GITHUB_RESULTS_TOKEN` with repository **Contents: Read and write**, and enable notebook access. The token is never added to notebooks, ZIP, CSV, or the public repository. New UTC folders avoid accidental overwrites. If the secret is absent, use the ZIP download cell.

In [ ]:
import requests
def github_token():
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_RESULTS_TOKEN")
    except Exception:
        return os.environ.get("GITHUB_RESULTS_TOKEN")
if UPLOAD_TO_GITHUB:
    token=github_token()
    if not token:
        print("GITHUB_RESULTS_TOKEN unavailable. GitHub upload skipped; results are stored locally:",OUT)
    else:
        print("Uploading to repository",REPO,"folder 1019/results/"+RUN_ID)
        files_to_upload=[p for p in sorted(OUT.iterdir()) if p.suffix.lower() in {".csv",".png",".json",".md"}]
        uploaded=[];failed=[]
        headers={"Authorization":"Bearer "+token,"Accept":"application/vnd.github+json",
                 "X-GitHub-Api-Version":"2022-11-28"}
        for file in tqdm(files_to_upload,desc="Saving experiment outputs to GitHub"):
            remote=f"1019/results/{RUN_ID}/{file.name}"
            url=f"https://api.github.com/repos/{REPO}/contents/{remote}"
            payload={"message":f"research: save {RUN_ID} result {file.name}",
                     "content":base64.b64encode(file.read_bytes()).decode("ascii"),
                     "branch":BRANCH}
            try:
                # A repeat of only this cell should be idempotent, not fail with HTTP 422.
                existing=requests.get(url,headers=headers,timeout=30)
                if existing.status_code==200:
                    payload["sha"]=existing.json()["sha"]
                elif existing.status_code!=404:
                    raise RuntimeError(f"GitHub preflight HTTP {existing.status_code}")
                response=requests.put(url,headers=headers,json=payload,timeout=90)
                if response.status_code not in (200,201):
                    raise RuntimeError(f"HTTP {response.status_code}; check token Contents write access or path collision.")
                uploaded.append(remote)
            except Exception as e:
                failed.append((file.name,str(e)));print("WARNING:",file.name,str(e))
        print(f"Uploaded {len(uploaded)}/{len(files_to_upload)} files.")
        if uploaded:print(f"https://github.com/{REPO}/tree/{BRANCH}/1019/results/{RUN_ID}")
        if failed:print("Some files failed. All remain in the local ZIP:",zip_path)
        del token,headers
else:
    print("UPLOAD_TO_GITHUB=False. GitHub upload skipped; all files remain in",OUT)

In [ ]:
# Run this cell if you want the CSV/PNG/JSON/README bundle on your computer.
try:
    from google.colab import files as colab_files
    colab_files.download(str(zip_path))
except ImportError:
    print("The output ZIP is stored at:",zip_path)